# Exploring Microsoft Qlib on Google Colab

This notebook provides a guided tour of [Microsoft Qlib](https://github.com/microsoft/qlib), an open-source platform for **quantitative research** and **trading**. It focuses on running within Google Colab so you can experiment quickly without local setup.

> **Tip:** Run cells from top to bottom in Colab. Sections marked as optional can be skipped if you only want to read the explanations.

## 1. Environment setup

These cells prepare Colab with the dependencies needed to explore Qlib locally. The installation pulls the published package from PyPI and clones the GitHub repository for reference.

* `pip install qlib`: installs the library and core dependencies.
* `git clone https://github.com/microsoft/qlib`: grabs the source code for browsing and reading.
* `pip install -r requirements.txt`: installs extra dependencies used by Qlib's examples (some are optional; feel free to skip if you only need the library).

> If you see warnings about `lightgbm`, they are expected because some deep-learning examples require it. You can add or remove extras depending on your experiment.

In [ ]:
# Install the published package and clone the repository
!pip -q install qlib
!git clone --depth 1 https://github.com/microsoft/qlib.git
%cd qlib
!pip -q install -r requirements.txt
%cd ..

## 2. Repository tour

The repository contains several key directories. The code cell below prints a short tree so you can connect the documentation to the source layout.

* `qlib/`: core Python package implementing data handling, model interfaces, backtesting, and utility components.
* `examples/`: runnable examples for forecasting, portfolio construction, and backtesting. They are the fastest way to learn typical workflows.
* `scripts/`: helper scripts to prepare datasets (e.g., Yahoo data), run benchmarks, and manage features.
* `docs/`: additional Sphinx documentation.
* `benchmarks/`: standardized benchmark configurations for research comparisons.

In [ ]:
import os
root = 'qlib'
keep = {'qlib', 'examples', 'scripts', 'docs', 'benchmarks'}
rows = []
for entry in sorted(os.listdir(root)):
    if entry.startswith('.') or entry in {'dist', 'build'}:
        continue
    path = os.path.join(root, entry)
    if entry in keep and os.path.isdir(path):
        children = sorted(os.listdir(path))[:10]
        rows.append(f"{entry}/ -> {', '.join(children)}")
print("
".join(rows))

## 3. Quickstart: initialize Qlib and load data

Qlib uses a data provider abstraction. For lightweight exploration on Colab, use the built-in Yahoo data initializer. The snippet below downloads a small daily dataset, initializes Qlib, and queries basic features.

* `qlib.init`: sets the data path and enables the provider.
* `D.features`: fetches a feature DataFrame for a universe of tickers.

> The initial download can take a few minutes depending on network speed.

In [ ]:
from qlib.data import D
import qlib
from qlib.config import REG_CN

# Initialize with a temporary directory inside Colab
data_dir = '/content/qlib_data'
qlib.init(provider_uri=data_dir, region=REG_CN)

# Prepare a small stock universe and pull close prices
market = 'csi300'
universe = D.list_instruments(market=market, start_time='2022-01-01', end_time='2022-03-01')[:5]
features = D.features(universe, ['$close', '$volume'], start_time='2022-01-01', end_time='2022-03-01')
features.head()

## 4. Running a forecasting example

The `examples` directory provides ready-to-run workflows. The following cell executes the **single-factor linear regression** example, which demonstrates data preparation, model training, and evaluation on a small dataset.

* Configuration files live under `examples/benchmarks/` and use YAML to define data loaders, models, and evaluation metrics.
* `qlib.init` should point to the data directory prepared earlier.

If you want a more advanced deep-learning example, inspect `examples/benchmarks/MLP` or `examples/benchmarks/GRU` and adjust the `--config` path accordingly.

In [ ]:
%cd qlib/examples
!python train.py --config benchmarks/Linear/train_cn.yaml --task benchmark --data_dir /content/qlib_data --experiment_name demo_linear
%cd /content

## 5. Inspecting model outputs

Training logs and artifacts are stored under `mlruns/` (via MLflow) and `output/` (via Qlib's experiment manager). Use the snippet below to list the generated results and inspect prediction files or evaluation metrics.

* `mlruns/`: contains MLflow runs with parameters, metrics, and artifacts.
* `output/`: stores prediction CSVs and evaluation reports.

You can load the CSVs directly into pandas to plot returns or compare strategies.

In [ ]:
import pandas as pd
import pathlib

run_root = pathlib.Path('qlib/examples/mlruns')
latest = max(run_root.iterdir(), key=lambda p: p.stat().st_mtime)
print('Latest MLflow run:', latest)

# Load one prediction file if it exists
preds = list(pathlib.Path('qlib/examples/output').glob('**/pred.pkl'))
if preds:
    df = pd.read_pickle(preds[0])
    display(df.head())
else:
    print('No predictions found yet. Check the example run output above.')

## 6. Where to dive deeper in the codebase

Here are key modules worth exploring in the cloned repository:

* **Data layer** (`qlib/data`): providers (`provider.py`), handlers (`dataset/handler.py`), and pipeline utilities for fetching and caching market data.
* **Model interfaces** (`qlib/model`): base classes and implementations for machine learning models (e.g., `base.py`, `pytorch_graph.py`).
* **Workflow** (`qlib/workflow`): experiment tracking, backtesting, and execution (`task/train.py`, `task/collector.py`).
* **Examples and benchmarks** (`examples/benchmarks`): configuration-driven experiments; start with `Linear` or `MLP` configs.
* **Scripts** (`scripts/data_collector`): tools for preparing data from Yahoo Finance or CSI market sources.

Use Colab's file browser or `!sed -n '1,120p file.py'` to read through any file directly in the notebook.

## 7. Next steps

* Try swapping the benchmark configuration to a neural model (e.g., `benchmarks/GRU/train_cn.yaml`).
* Explore portfolio construction via `examples/port_analysis` or `benchmarks/PortAna`.
* Read the official documentation in `docs/` for architectural diagrams and concept overviews.

Happy researching!